In [1]:
import pandas as pd
import numpy as np
from scipy import stats

In [3]:
import zipfile
import os

with zipfile.ZipFile("engineered_data.zip", "r") as zip_ref:
    zip_ref.extractall("engineered")

print(os.listdir("engineered"))

['order_items_engineered.csv', 'orders_engineered.csv', 'products_engineered.csv', 'reviews_engineered.csv', 'customers_engineered.csv', 'sellers_engineered.csv']


In [ ]:
orders = pd.read_csv("../data/engineered/orders_engineered.csv")
order_items = pd.read_csv("../data/engineered/order_items_engineered.csv")
products = pd.read_csv("../data/engineered/products_engineered.csv")

In [5]:
orders["delivery_status"] = np.where(
    orders["delivery_delay_days"] > 0,
    "Delayed",
    "On Time"
)

delayed_reviews = orders.loc[
    orders["delivery_status"] == "Delayed",
    "review_score"
].dropna()

on_time_reviews = orders.loc[
    orders["delivery_status"] == "On Time",
    "review_score"
].dropna()

t_stat, p_value = stats.ttest_ind(
    delayed_reviews,
    on_time_reviews,
    equal_var=False
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: -82.68184427423601
P-value: 0.0


In [6]:
anova_data = (
    order_items
    .merge(
        products[
            ["product_id", "product_category_name_english"]
        ],
        on="product_id",
        how="left"
    )
)

anova_data["product_category_name_english"] = (
    anova_data["product_category_name_english"]
    .fillna("unknown")
)

category_groups = [
    group["price"].dropna().values
    for _, group in anova_data.groupby(
        "product_category_name_english"
    )
    if len(group["price"].dropna()) > 1
]

f_stat, p_value = stats.f_oneway(*category_groups)

print("F-statistic:", f_stat)
print("P-value:", p_value)

F-statistic: 188.60771157282187
P-value: 0.0


In [8]:
from google.colab import files

uploaded = files.upload()

Saving olist_order_payments_dataset.csv to olist_order_payments_dataset.csv


In [ ]:
payments = pd.read_csv(
    "../data/cleaned/payments.csv",
    engine="python"
)

orders_status = orders[
    ["order_id", "order_status"]
]

chi_data = payments.merge(
    orders_status,
    on="order_id",
    how="inner"
)

contingency_table = pd.crosstab(
    chi_data["payment_type"],
    chi_data["order_status"]
)

chi2_stat, chi_p_value, dof, expected = stats.chi2_contingency(
    contingency_table
)

print("Chi-square statistic:", chi2_stat)
print("Degrees of freedom:", dof)
print("P-value:", chi_p_value)

Chi-square statistic: 677.0831369637666
Degrees of freedom: 28
P-value: 1.2047397488650834e-124


In [ ]:
payments = pd.read_csv(
    "../data/cleaned/payments.csv",
    engine="python"
)

orders_status = orders[
    ["order_id", "order_status"]
]

chi_data = payments.merge(
    orders_status,
    on="order_id",
    how="inner"
)

contingency_table = pd.crosstab(
    chi_data["payment_type"],
    chi_data["order_status"]
)

chi2_stat, chi_p_value, dof, expected = stats.chi2_contingency(
    contingency_table
)

print("Chi-square statistic:", chi2_stat)
print("Degrees of freedom:", dof)
print("P-value:", chi_p_value)

Chi-square statistic: 677.0831369637666
Degrees of freedom: 28
P-value: 1.2047397488650834e-124


In [11]:
t_stat, t_p_value = stats.ttest_ind(
    delayed_reviews,
    on_time_reviews,
    equal_var=False
)

f_stat, f_p_value = stats.f_oneway(*category_groups)

results = pd.DataFrame({
    "Test": [
        "Independent Two-Sample T-Test",
        "One-Way ANOVA",
        "Chi-Square Test"
    ],
    "Statistic": [
        t_stat,
        f_stat,
        chi2_stat
    ],
    "P-Value": [
        t_p_value,
        f_p_value,
        chi_p_value
    ],
    "Decision": [
        "Reject H0" if t_p_value < 0.05 else "Fail to Reject H0",
        "Reject H0" if f_p_value < 0.05 else "Fail to Reject H0",
        "Reject H0" if chi_p_value < 0.05 else "Fail to Reject H0"
    ]
})

results

,Test,Statistic,P-Value,Decision
0,Independent Two-Sample T-Test,-82.681844,0.000000e+00,Reject H0
1,One-Way ANOVA,188.607712,0.000000e+00,Reject H0
2,Chi-Square Test,677.083137,1.204740e-124,Reject H0


In [12]:
print("T-Test Interpretation:")
if t_p_value < 0.05:
    print("There is a statistically significant difference in review scores between delayed and on-time orders.")
else:
    print("There is no statistically significant difference in review scores between delayed and on-time orders.")

print("\nANOVA Interpretation:")
if f_p_value < 0.05:
    print("There is a statistically significant difference in average product price across product categories.")
else:
    print("There is no statistically significant difference in average product price across product categories.")

print("\nChi-Square Interpretation:")
if chi_p_value < 0.05:
    print("Payment method and order status are statistically associated.")
else:
    print("Payment method and order status are not statistically associated.")

T-Test Interpretation:
There is a statistically significant difference in review scores between delayed and on-time orders.

ANOVA Interpretation:
There is a statistically significant difference in average product price across product categories.

Chi-Square Interpretation:
Payment method and order status are statistically associated.


In [13]:
results.to_csv("statistical_results.csv", index=False)

print("Statistical analysis completed successfully!")

Statistical analysis completed successfully!
